In [1]:
# ============================================================
# EXERCICE : COMPARAISON DE MODÈLES DE TRADUCTION ANGLAIS → FRANÇAIS
# ============================================================
"""
Contexte
Vous êtes consultant·e en traduction automatique.
Un client vous demande d'évaluer la qualité de plusieurs systèmes de traduction automatique
(modèles Hugging Face et systèmes en ligne) sur des textes de genres variés, traduits de l'anglais vers le français.
Votre mission se déroule en trois étapes.

Étape 1 : Comparaison des traductions
Objectif : Comparer les traductions produites par :

Des modèles Hugging Face (via le notebook fourni) :
T5-small
MarianMT (Helsinki-NLP)
mBART-50
NLLB-200-distilled

Des systèmes de traduction en ligne :
Google Translate
DeepL
(Optionnel) Reverso, Bing Translator, Systran

Étape 2 : Analyse critique des traductions
Objectif : Commenter les traductions en vous appuyant sur :

A. La stratégie de traduction
Questions guidées :
Le système a-t-il conservé les références culturelles ou les a-t-il adaptées
Le registre (familier, soutenu, archaïque) est-il préservé ?
Les idiomes sont-ils traduits littéralement ou fonctionnellement ?

B. Les erreurs de traduction
Identifiez et classez les erreurs observées

C. Vos propres exemples
Objectif : « Piéger » la machine et proposer des améliorations.

Étape 3 : Rapport analytique
Objectif : Rédiger un court rapport

"""

"\nContexte\nVous êtes consultant·e en traduction automatique.\nUn client vous demande d'évaluer la qualité de plusieurs systèmes de traduction automatique\n(modèles Hugging Face et systèmes en ligne) sur des textes de genres variés, traduits de l'anglais vers le français.\nVotre mission se déroule en trois étapes.\n\nÉtape 1 : Comparaison des traductions\nObjectif : Comparer les traductions produites par :\n\nDes modèles Hugging Face (via le notebook fourni) :\nT5-small\nMarianMT (Helsinki-NLP)\nmBART-50\nNLLB-200-distilled\n\nDes systèmes de traduction en ligne :\nGoogle Translate\nDeepL\n(Optionnel) Reverso, Bing Translator, Systran\n\nÉtape 2 : Analyse critique des traductions\nObjectif : Commenter les traductions en vous appuyant sur :\n\nA. La stratégie de traduction\nQuestions guidées :\nLe système a-t-il conservé les références culturelles ou les a-t-il adaptées\nLe registre (familier, soutenu, archaïque) est-il préservé ?\nLes idiomes sont-ils traduits littéralement ou fonctio

In [2]:
# --- Installation des dépendances ---
# !pip install transformers torch sacrebleu datasets sentencepiece

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

/home/cam/git/pluriTAL/traduction-automatique/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# ============================================================
# PARTIE 1 : EXEMPLES DE TEST (5 GENRES)
# ============================================================

# --- 1. TWEET (registre familier, ironie, hashtags) ---
tweet_examples = [
    "This meeting could've been an email 🙄 #CorporateLife",
    "Just spent 3 hours on a 5-minute task. Living the dream! 😅",
    "im so done w/ this rn 😩 #MondayMood"
]

# --- 2. BIBLE (archaïsme, style solennel) ---
bible_examples = [
    "The spirit is willing, but the flesh is weak.",
    "In the beginning was the Word, and the Word was with God, and the Word was God.",
    "Yea, though I walk through the valley of the shadow of death, I will fear no evil."
]

# --- 3. SHAKESPEARE (versification, archaïsme, jeu de mots) ---
shakespeare_examples = [
    "And yet, by heaven, I think my love as rare / As any she belied with false compare.",
    "Whoever hath her wish, thou hast thy Will",
    "That time of year thou mayst in me behold / When yellow leaves, or none, or few, do hang / Upon those boughs which shake against the cold."
]

# --- 4. CONTRAT JURIDIQUE (terminologie précise, syntaxe complexe) ---
legal_examples = [
    "The Lessee shall indemnify and hold harmless the Lessor from and against any and all claims.",
    "This Agreement shall be governed by and construed in accordance with the laws of the State of New York.",
    "The parties hereto agree that any dispute arising hereunder shall be resolved by binding arbitration."
]

# --- 5. NOTICE TECHNIQUE (vocabulaire standardisé, syntaxe simple) ---
technical_examples = [
    "Tighten screw A to 5 Nm.",
    "Do not immerse the device in water.",
    "Press the power button for 3 seconds to turn on the device."
]

# --- 6. Slogans ---
slogan_examples = [
    "Be a goat. (Columbia Sportswear)",
    "Taste. Not waist. (Weight Watchers)",
    "Finger Lickin' good. (KFC)"
]

# Regroupement
all_examples = {
    "Tweet": tweet_examples,
    "Bible": bible_examples,
    "Shakespeare": shakespeare_examples,
    "Juridique": legal_examples,
    "Technique": technical_examples,
    "Slogans": slogan_examples
}

print("Exemples chargés :")
for genre, examples in all_examples.items():
    print(f"  - {genre} : {len(examples)} exemples")


Exemples chargés :
  - Tweet : 3 exemples
  - Bible : 3 exemples
  - Shakespeare : 3 exemples
  - Juridique : 3 exemples
  - Technique : 3 exemples
  - Slogans : 3 exemples


In [4]:

# ============================================================
# PARTIE 2 : CHARGEMENT DES MODÈLES
# ============================================================

# Liste de modèles à comparer
MODELS = {
    "T5-small": "google-t5/t5-small",
    "MarianMT (Helsinki)": "Helsinki-NLP/opus-mt-en-fr",
    "mBART-50": "facebook/mbart-large-50-many-to-many-mmt",
    "NLLB-200-distilled": "facebook/nllb-200-distilled-600M",
}

# Chargement des modèles
models = {}

for name, checkpoint in MODELS.items():
    print(f"\nChargement de {name} ({checkpoint})...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
        models[name] = {"tokenizer": tokenizer, "model": model}
        print(f"  ✓ {name} chargé avec succès")
    except Exception as e:
        print(f"  ✗ Erreur avec {name} : {e}")


Chargement de T5-small (google-t5/t5-small)...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1688.46it/s]


  ✓ T5-small chargé avec succès

Chargement de MarianMT (Helsinki) (Helsinki-NLP/opus-mt-en-fr)...


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 6413.38it/s]


  ✓ MarianMT (Helsinki) chargé avec succès

Chargement de mBART-50 (facebook/mbart-large-50-many-to-many-mmt)...


Loading weights: 100%|██████████| 516/516 [00:00<00:00, 1566.91it/s]


  ✓ mBART-50 chargé avec succès

Chargement de NLLB-200-distilled (facebook/nllb-200-distilled-600M)...


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 5847.93it/s]

  ✓ NLLB-200-distilled chargé avec succès


In [5]:

# ============================================================
# PARTIE 3 : FONCTIONS D'INFÉRENCE
# ============================================================

def translate_t5(text, model, tokenizer):
    """Traduction avec T5 (nécessite un préfixe)."""
    prefix = "translate English to French: "
    input_text = prefix + text
    inputs = tokenizer(input_text, return_tensors="pt").input_ids
    outputs = model.generate(inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def translate_marian(text, model, tokenizer):
    """Traduction avec MarianMT (pas de préfixe)."""
    inputs = tokenizer(text, return_tensors="pt").input_ids
    outputs = model.generate(inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def translate_mbart(text, model, tokenizer):
    """Traduction avec mBART (nécessite un code de langue)."""
    tokenizer.src_lang = "en_XX"
    inputs = tokenizer(text, return_tensors="pt").input_ids
    outputs = model.generate(
        inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["fr_XX"],
        max_new_tokens=128
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def translate_nllb(text, model, tokenizer):
    """Traduction avec NLLB (nécessite un code de langue)."""
    tokenizer.src_lang = "eng_Latn"
    inputs = tokenizer(text, return_tensors="pt").input_ids
    outputs = model.generate(
        inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),
        max_new_tokens=128
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Dictionnaire de fonctions par modèle
TRANSLATE_FUNCS = {
    "T5-small": translate_t5,
    "MarianMT (Helsinki)": translate_marian,
    "mBART-50": translate_mbart,
    "NLLB-200-distilled": translate_nllb,
}

In [6]:
# ============================================================
# PARTIE 4 : INFÉRENCE ET COMPARAISON
# ============================================================

def translate_all(text):
    """Traduit un texte avec tous les modèles chargés."""
    results = {}
    for name, data in models.items():
        try:
            func = TRANSLATE_FUNCS[name]
            translation = func(text, data["model"], data["tokenizer"])
            results[name] = translation
        except Exception as e:
            results[name] = f"[ERREUR: {e}]"
    return results


# Exécution sur tous les exemples
print("\n" + "=" * 80)
print("RÉSULTATS DE TRADUCTION")
print("=" * 80)

for genre, examples in all_examples.items():
    print(f"\n{'─' * 80}")
    print(f"GENRE : {genre}")
    print(f"{'─' * 80}")

    for i, text in enumerate(examples, 1):
        print(f"\n[{i}] EN : {text}")
        translations = translate_all(text)
        for model_name, translation in translations.items():
            print(f"    {model_name:25s} : {translation}")


RÉSULTATS DE TRADUCTION

────────────────────────────────────────────────────────────────────────────────
GENRE : Tweet
────────────────────────────────────────────────────────────────────────────────

[1] EN : This meeting could've been an email 🙄 #CorporateLife


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  : Cette réunion aurait pu être un courriel  #CorporateLife
    MarianMT (Helsinki)       : Cette réunion aurait pu être un e-mail #CorporateLife
    mBART-50                  : Cette rencontre aurait pu être un courriel  😉 #CorporateLife
    NLLB-200-distilled        : Cette réunion aurait pu être un courriel 🙄 #CorporateLife

[2] EN : Just spent 3 hours on a 5-minute task. Living the dream! 😅


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

    T5-small                  :                                                         
    MarianMT (Helsinki)       : Je viens de passer 3 heures sur une tâche de 5 minutes. Vivre le rêve!
    mBART-50                  : J'ai tout juste passé 3 heures sur une tâche de 5 minutes.
    NLLB-200-distilled        : J'ai passé 3 heures à une tâche de 5 minutes.

[3] EN : im so done w/ this rn 😩 #MondayMood


...

    T5-small                  : tre une chèvre (Columbia Sportswear)
    MarianMT (Helsinki)       : Soyez une chèvre. (Columbia Sportswear)
    mBART-50                  : Soyez une chèvre. (Columbia Sportswear)
    NLLB-200-distilled        : Sois une chèvre.

[2] EN : Taste. Not waist. (Weight Watchers)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  : Taste, pas waist. (Veilleurs de poids)
    MarianMT (Helsinki)       : Pas la taille.
    mBART-50                  : Taste. Non Taille. (Vérificateurs de la taille)
    NLLB-200-distilled        : Le goût, pas la taille.

[3] EN : Finger Lickin' good. (KFC)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  :  l'aise, il y a eu un accroissement de la pression et de la pression.
    MarianMT (Helsinki)       : C'est bon, le doigt.
    mBART-50                  : Finger Lickin' bon. (KFC)
    NLLB-200-distilled        : Le "Démarrer les doigts" va bien.


In [7]:
# ============================================================
# PARTIE 5 : ANALYSE QUANTITATIVE (BLEU)
# ============================================================

# Si vous avez des traductions de référence, vous pouvez calculer le BLEU
# Exemple avec sacrebleu :

import sacrebleu

# Exemple de références (à compléter par l'enseignant)
references = {
    "The spirit is willing, but the flesh is weak.": [
        "L'esprit est bien disposé, mais la chair est faible."
    ],
    "Shall I compare thee to a summer's day?": [
        "Te comparerai-je à un jour d'été ?"
    ],
    "Tighten screw A to 5 Nm.": [
        "Serrer la vis A à 5 Nm."
    ],
}

def compute_bleu(hypothesis, reference):
    """Calcule le score BLEU."""
    bleu = sacrebleu.corpus_bleu([hypothesis], [reference])
    return bleu.score

print("\n" + "=" * 80)
print("SCORES BLEU (sur exemples avec référence)")
print("=" * 80)

for text, refs in references.items():
    print(f"\nEN : {text}")
    translations = translate_all(text)
    for model_name, translation in translations.items():
        if not translation.startswith("[ERREUR"):
            score = compute_bleu(translation, refs)
            print(f"    {model_name:25s} : BLEU = {score:.2f}")


SCORES BLEU (sur exemples avec référence)

EN : The spirit is willing, but the flesh is weak.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  : BLEU = 75.17
    MarianMT (Helsinki)       : BLEU = 75.17
    mBART-50                  : BLEU = 75.17
    NLLB-200-distilled        : BLEU = 100.00

EN : Shall I compare thee to a summer's day?


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  : BLEU = 9.29
    MarianMT (Helsinki)       : BLEU = 12.22
    mBART-50                  : BLEU = 51.70
    NLLB-200-distilled        : BLEU = 51.70

EN : Tighten screw A to 5 Nm.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    T5-small                  : BLEU = 0.00
    MarianMT (Helsinki)       : BLEU = 53.29
    mBART-50                  : BLEU = 25.85
    NLLB-200-distilled        : BLEU = 84.09
